In [1]:
import anndata as ad
import gseapy as gs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pertpy as pt
import scanpy as sc
import seaborn as sns
from scxmatch import *
from statsmodels.stats.multitest import multipletests

will use the CPU to calculate the distance matrix.


In [2]:
np.random.seed(43)

In [3]:
group_by = "perturbation"
reference = "control"

In [4]:
hallmark = gs.get_library('MSigDB_Hallmark_2020')

2026-05-06 15:44:49 | [INFO] Downloading and generating Enrichr library gene sets...


In [5]:
adata = ad.read_h5ad("../data/LSI_MimitouSmibert2021.hdf5") # this is not log1p transformed and contains all genes
adata = adata[adata.obs["perturbation"].dropna().index].copy()
#adata.obs[group_by] = adata.obs[group_by].astype(str)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata)
adata.var.rename({"Unnamed: 0": "gene"}, axis=1, inplace=True)
adata.var.set_index("gene", inplace=True)

/data/bionets/je30bery/conda/envs/gseapy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/data/bionets/je30bery/conda/envs/gseapy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [6]:
test_groups = adata.obs["perturbation"].unique()

In [7]:
group_counts = adata.obs[group_by].value_counts()
min_count = group_counts.min()

print(f"Minimum group size: {min_count}", flush=True)

sampled_indices = []
relative_support_dict = dict()

for g in group_counts.index:
    idx = np.where(adata.obs[group_by] == g)[0]
    sampled = np.random.choice(idx, min_count, replace=False)
    sampled_indices.append(sampled)
    relative_support_dict[g] = len(sampled) / len(idx)

sampled_indices = np.concatenate(sampled_indices)
adata_balanced = adata[sampled_indices].copy()

Minimum group size: 621


In [8]:
n_comps = min(50, adata_balanced.shape[1] - 1, adata_balanced.shape[0] - 1)
sc.pp.pca(adata_balanced, n_comps=n_comps)
distance_test = pt.tl.DistanceTest("edistance", n_perms=10000, obsm_key="X_pca")
tab = distance_test(adata_balanced, groupby="perturbation", contrast="control")
tab

Output()

,distance,pvalue,significant,pvalue_adj,significant_adj
ZAP70,0.930593,0.0001,True,0.0006,True
CD4,0.085171,0.0001,True,0.0006,True
control,0.000000,1.0000,False,1.0000,False
NFKB2,0.084160,0.0001,True,0.0006,True
CD3ECD4,1.564950,0.0001,True,0.0006,True
CD3E,1.822393,0.0001,True,0.0006,True


In [9]:
n_comps = min(50, adata_balanced.shape[1] - 1, adata_balanced.shape[0] - 1)
distance_test = pt.tl.DistanceTest("edistance", n_perms=20000, obsm_key="X_pca")
tab = distance_test(adata_balanced, groupby="perturbation", contrast="control")
tab

Output()

,distance,pvalue,significant,pvalue_adj,significant_adj
ZAP70,0.930593,0.00005,True,0.0003,True
CD4,0.085171,0.00005,True,0.0003,True
control,0.000000,1.00000,False,1.0000,False
NFKB2,0.084160,0.00005,True,0.0003,True
CD3ECD4,1.564950,0.00005,True,0.0003,True
CD3E,1.822393,0.00005,True,0.0003,True


In [10]:
n_comps = min(50, adata.shape[1] - 1, adata.shape[0] - 1)
sc.pp.pca(adata, n_comps=n_comps)
distance_test = pt.tl.DistanceTest("edistance", n_perms=10000, obsm_key="X_pca")
tab = distance_test(adata, groupby="perturbation", contrast="control")
tab

Output()

,distance,pvalue,significant,pvalue_adj,significant_adj
NFKB2,0.073150,0.0001,True,0.0006,True
CD4,0.040700,0.0001,True,0.0006,True
control,0.000000,1.0000,False,1.0000,False
ZAP70,0.847151,0.0001,True,0.0006,True
CD3E,1.757345,0.0001,True,0.0006,True
CD3ECD4,1.513137,0.0001,True,0.0006,True


In [11]:
n_comps = min(50, adata.shape[1] - 1, adata.shape[0] - 1)
distance_test = pt.tl.DistanceTest("edistance", n_perms=20000, obsm_key="X_pca")
tab = distance_test(adata, groupby="perturbation", contrast="control")
tab

Output()

,distance,pvalue,significant,pvalue_adj,significant_adj
NFKB2,0.073150,0.00005,True,0.0003,True
CD4,0.040700,0.00005,True,0.0003,True
control,0.000000,1.00000,False,1.0000,False
ZAP70,0.847151,0.00005,True,0.0003,True
CD3E,1.757345,0.00005,True,0.0003,True
CD3ECD4,1.513137,0.00005,True,0.0003,True


In [12]:
adata = ad.read_h5ad("../data/LSI_MimitouSmibert2021.hdf5") # this is not log1p transformed and contains all genes
key = 'TNF-alpha Signaling via NF-kB'
name = 'MSigDB_Hallmark_2020'

adata = adata[adata.obs["perturbation"].dropna().index].copy()
sc.pp.log1p(adata)
#sc.pp.highly_variable_genes(adata)
adata.var.rename({"Unnamed: 0": "gene"}, axis=1, inplace=True)
adata.var.set_index("gene", inplace=True)
subset = adata[:, np.isin(adata.var_names, hallmark[key])].copy()

/data/bionets/je30bery/conda/envs/gseapy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/data/bionets/je30bery/conda/envs/gseapy/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [13]:
rows = list()

In [14]:
for test_group in test_groups:
    if test_group != "control":
        p, z, s = test(adata=subset, reference=reference, test_group=test_group, k="full", metric="sqeuclidean", rank=False, group_by=group_by)
        rows.append([f"{name} {key}", subset.shape[1], len(hallmark[key]), test_group, p, z, s])

using CPU to calculate distance matrix.
creating distance graph with 2136 samples
counting cross matches.


NameError: name 'geneset' is not defined

In [ ]:
results = pd.DataFrame(rows, columns=["Gene set", "#Genes used", "#Genes total", "Test group (knockout)", "P", "Z", "S"])
results["P_adj"] = multipletests(results["P"], method="fdr_bh")[1]

In [ ]:
group_counts = subset.obs[group_by].value_counts()
min_count = group_counts.min()

print(f"Minimum group size: {min_count}", flush=True, file=sys.stderr)

sampled_indices = []
relative_support_dict = dict()

for g in group_counts.index:
    idx = np.where(subset.obs[group_by] == g)[0]
    sampled = np.random.choice(idx, min_count, replace=False)
    sampled_indices.append(sampled)
    relative_support_dict[g] = len(sampled) / len(idx)

sampled_indices = np.concatenate(sampled_indices)

In [ ]:
subset_balanced = subset[sampled_indices].copy()

In [ ]:
n_comps = min(50, subset.shape[1] - 1, subset.shape[0] - 1)
sc.pp.pca(subset_balanced, n_comps=n_comps)
distance_test = pt.tl.DistanceTest("edistance", n_perms=10000, obsm_key="X_pca")
tab = distance_test(subset_balanced, groupby="perturbation", contrast="control")

In [ ]:
pd.options.display.float_format = "{:,.1e}".format
results.sort_values("Test group (knockout)")

In [ ]:
tab